# Task 2: Exploratory Data Analysis — Movie Analytics

CodeAlpha Data Analytics Internship

This notebook explores the dataset built in Task 1 (`movies_raw.csv`): the highest-grossing
films of all time, enriched with genre, IMDb rating, runtime, director, and more.

Questions we're digging into:
- What shape is this data in, and what needs cleaning?
- Which genres dominate, and has that changed over time?
- Does IMDb rating actually predict box office success?
- What are the standout outliers — critical darlings that didn't make money, and
  blockbusters critics didn't love?

Run every cell top to bottom. The printed output under each cell *is* the finding —
copy the numbers that surprise you into your write-up/LinkedIn post.


In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## 1. Load the data

In [ ]:
df = pd.read_csv("data/movies_raw.csv")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


In [ ]:
df.info()


## 2. Cleaning

Straight out of the scraper, a few columns aren't analysis-ready:
- `worldwide_gross` has `$`, commas, and occasional stray currency-code artifacts from
  Wikipedia's footnotes (e.g. `SM$1,922,598,800`) — we just need the digits.
- `runtime` is a string like `"162 min"` — need the number.
- `imdb_rating` / `imdb_votes` are strings, and OMDb returns `"N/A"` for missing data.
- `genre` is a comma-separated string — we'll need it split out for genre-level analysis.


In [ ]:
def to_number(value):
    """Strip everything except digits and turn into a number, or NaN if nothing's left."""
    if pd.isna(value):
        return np.nan
    digits = re.sub(r"[^0-9]", "", str(value))
    return float(digits) if digits else np.nan

def runtime_to_minutes(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"(\d+)", str(value))
    return int(match.group(1)) if match else np.nan

df["gross_usd"] = df["worldwide_gross"].apply(to_number)
df["runtime_min"] = df["runtime"].apply(runtime_to_minutes)
df["imdb_rating"] = pd.to_numeric(df["imdb_rating"], errors="coerce")
df["imdb_votes"] = df["imdb_votes"].apply(to_number)
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["decade"] = (df["year"] // 10 * 10).astype("Int64")

# Split genre into a list for genre-level analysis later
df["genre_list"] = df["genre"].apply(
    lambda g: [x.strip() for x in g.split(",")] if pd.notna(g) else []
)

df[["title", "gross_usd", "runtime_min", "imdb_rating", "imdb_votes", "decade"]].head()


### A quick sanity check on the numbers

No film has ever grossed anywhere near $3 billion worldwide. If cleaning produced
anything above that, it's a leftover scraping artifact (usually a footnote reference
number that merged into the digits) rather than a real figure — better to flag it than
silently trust it.


In [ ]:
suspect = df[df["gross_usd"] > 3e9]
if len(suspect) > 0:
    print(f"Flagged {len(suspect)} row(s) with an implausible gross - setting to NaN:")
    print(suspect[["title", "worldwide_gross", "gross_usd"]].to_string(index=False))
    df.loc[df["gross_usd"] > 3e9, "gross_usd"] = np.nan
else:
    print("No implausible gross values found.")


## 3. How much OMDb enrichment actually worked?

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({"missing_count": missing, "missing_%": missing_pct})


In [ ]:
match_rate = 100 - df["imdb_rating"].isna().mean() * 100
print(f"OMDb successfully matched {match_rate:.1f}% of the {len(df)} scraped titles.")
print("The rest are likely title-mismatch cases (sequels, re-releases, special characters in titles).")


### Rows worth double-checking

`scrape_movies.py` now flags any match with a suspiciously low vote count as
`match_uncertain`. Worth a quick manual glance at these before trusting them fully.


In [ ]:
if "match_uncertain" in df.columns:
    uncertain = df[df["match_uncertain"] == True]
    print(f"{len(uncertain)} row(s) flagged as uncertain matches:\n")
    print(uncertain[["title", "year", "imdb_rating", "imdb_votes"]].to_string(index=False))
else:
    print("Re-run the updated scrape_movies.py to get the match_uncertain column.")


## 4. Genre trends

In [ ]:
genre_counts = df.explode("genre_list")["genre_list"].value_counts()
print("Most common genres among the highest-grossing films of all time:\n")
print(genre_counts.head(10))


In [ ]:
genre_by_decade = (
    df.explode("genre_list")
    .dropna(subset=["decade"])
    .groupby(["decade", "genre_list"])
    .size()
    .reset_index(name="count")
)
top_genre_per_decade = genre_by_decade.loc[genre_by_decade.groupby("decade")["count"].idxmax()]
print("Top genre by decade:\n")
print(top_genre_per_decade[["decade", "genre_list", "count"]].to_string(index=False))


## 5. Does IMDb rating predict box office success?

In [ ]:
numeric_cols = ["gross_usd", "imdb_rating", "imdb_votes", "runtime_min"]
corr = df[numeric_cols].corr()
print(corr.round(2))


In [ ]:
# Quick exploratory plot — not the polished version, just a gut check.
# The refined, presentation-ready charts happen in Task 3.
plt.figure(figsize=(7, 5))
plt.scatter(df["imdb_rating"], df["gross_usd"] / 1e9, alpha=0.6)
plt.xlabel("IMDb Rating")
plt.ylabel("Worldwide Gross ($ Billions)")
plt.title("Rating vs. Box Office — quick look")
plt.tight_layout()
plt.show()


A weak or near-zero correlation here isn't a bug — it's a genuinely interesting finding.
It suggests box office success is driven far more by franchise recognition, marketing
budget, and release timing than by how "good" a film is rated. Worth calling out
explicitly in your write-up rather than glossing over.


## 6. Outliers — the interesting exceptions

In [ ]:
print("Highest-grossing films with surprisingly LOW ratings (below 6.5):\n")
low_rated_hits = df[df["imdb_rating"] < 6.5].sort_values("gross_usd", ascending=False)
print(low_rated_hits[["title", "year", "gross_usd", "imdb_rating"]].head(10).to_string(index=False))


In [ ]:
print("Critically loved films (rating 8+) that made comparatively LESS money:\n")
loved_not_blockbusters = df[df["imdb_rating"] >= 8.0].sort_values("gross_usd", ascending=True)
print(loved_not_blockbusters[["title", "year", "gross_usd", "imdb_rating"]].head(10).to_string(index=False))


## 7. Save the cleaned dataset for Task 3

In [ ]:
clean_cols = [
    "rank", "title", "year", "decade", "gross_usd", "genre", "genre_list",
    "imdb_rating", "imdb_votes", "runtime_min", "director", "language",
    "country", "box_office_omdb", "awards",
]
clean_cols = [c for c in clean_cols if c in df.columns]
df[clean_cols].to_csv("data/movies_clean.csv", index=False)
print(f"Saved {len(df)} cleaned rows to data/movies_clean.csv")


## Summary — fill this in with your actual printed numbers

- **Data quality:** OMDb matched roughly `<match_rate>%` of scraped titles.
- **Genre trend:** `<top genre>` dominates the all-time list; by decade, the leading genre
  shifted from `<older decade genre>` toward `<recent decade genre>`.
- **Rating vs. gross:** correlation was `<corr value>` — essentially confirming that
  rating alone is a weak predictor of box office success.
- **Standout outlier:** `<lowest-rated blockbuster>` grossed the most despite one of the
  lowest ratings in the dataset; `<highest-rated under-performer>` was critically loved
  but made comparatively little.

This becomes the backbone of the Task 3 visualizations and the write-up/LinkedIn post
for this stage.
